In [ ]:
import csv
import time
import logging
from pathlib import Path
from urllib.parse import urlparse
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/9asset')
START_URL = "https://www.9asset.com/%E0%B8%82%E0%B8%B2%E0%B8%A2/%E0%B8%AD%E0%B8%AA%E0%B8%B1%E0%B8%87%E0%B8%AB%E0%B8%B2/%E0%B8%81%E0%B8%A3%E0%B8%B8%E0%B8%87%E0%B9%80%E0%B8%97%E0%B8%9E%E0%B8%AF"
OUTPUT_CSV_FILE = BASE_DIR / '9asset_listing_urls.csv'

MAX_PAGES = 10
PAGE_TIMEOUT = 15

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


def setup_optimized_driver() -> uc.Chrome:
    """Configures Chrome for maximum URL scraping performance."""
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.default_content_setting_values.notifications": 2,
    }
    options.add_experimental_option("prefs", prefs)
    
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    
    return uc.Chrome(options=options, version_main=145)


def extract_links(driver: uc.Chrome, base_url: str) -> list[str]:
    """Extracts property URLs using JS, handling Thai URL encoding (%E0...)."""
    js = """
    return [...new Set(
        Array.from(document.querySelectorAll('a.MuiPaper-root'))
        .map(a => a.getAttribute('href'))
        .filter(h => h && (h.includes('/ขาย/') || h.includes('/%E0%B8%82%E0%B8%B2%E0%B8%A2/')))
    )];
    """
    hrefs = driver.execute_script(js)
    return [f"{base_url}{h}" if h.startswith("/") else h for h in hrefs]


def scroll_to_bottom(driver: uc.Chrome):
    """Scrolls down to trigger any lazy-loaded content or pagination buttons."""
    last_h = 0
    for _ in range(6):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.5)
        h = driver.execute_script("return document.body.scrollHeight")
        if h == last_h:
            break
        last_h = h


def click_next_page(driver: uc.Chrome, wait: WebDriverWait) -> bool:
    """Finds and clicks the 'Next' pagination button using specific Ant Design SVG/Class logic."""
    try:
        xpath = "//button[contains(@class, 'ant-pagination-item-link') and .//span[contains(@class, 'anticon-right')]]"
        next_btn = wait.until(EC.presence_of_element_located((By.XPATH, xpath)))
        
        if next_btn.get_attribute("disabled"):
            return False
        try:
            parent_li = next_btn.find_element(By.XPATH, "..")
            if "disabled" in parent_li.get_attribute("class"):
                return False
        except NoSuchElementException:
            pass

        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_btn)
        time.sleep(0.5)
        driver.execute_script("arguments[0].click();", next_btn)
        return True
    except (TimeoutException, NoSuchElementException, ElementClickInterceptedException):
        return False


def get_first_listing_href(driver: uc.Chrome) -> str:
    """Retrieves the href of the first listing to monitor SPA state changes."""
    js = "return document.querySelector('a.MuiPaper-root')?.getAttribute('href') || '';"
    return driver.execute_script(js)


def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)
    
    driver = setup_optimized_driver()
    wait = WebDriverWait(driver, PAGE_TIMEOUT)
    parsed_url = urlparse(START_URL)
    base_url = f"{parsed_url.scheme}://{parsed_url.netloc}"
    
    all_urls = set()
    current_page = 1

    try:
        logging.info(f"Starting scrape at: {START_URL}")
        driver.get(START_URL)

        while current_page <= MAX_PAGES:
            logging.info(f"Processing Page {current_page}/{MAX_PAGES}...")
            
            try:
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a.MuiPaper-root')))
            except TimeoutException:
                logging.warning(f"Timeout waiting for elements on page {current_page}. Stopping.")
                break 

            scroll_to_bottom(driver)
            
            new_links = extract_links(driver, base_url)
            if not new_links:
                logging.info("No links found on this page. Stopping.")
                break
                
            all_urls.update(new_links)
            logging.info(f"Found {len(new_links)} listings (Total unique: {len(all_urls)})")

            if current_page >= MAX_PAGES:
                logging.info(f"Reached target max pages ({MAX_PAGES}).")
                break

            current_first_href = get_first_listing_href(driver)
            current_url = driver.current_url
            
            if not click_next_page(driver, wait):
                logging.info("No 'Next' button found or it is disabled. Reached the last page.")
                break
                
            try:
                # รอจนกว่าหน้าเว็บจะอัปเดตข้อมูลใหม่ (URL เปลี่ยน หรือ ข้อมูลการ์ดใบแรกเปลี่ยนไป)
                wait.until(
                    lambda d: d.current_url != current_url or get_first_listing_href(d) != current_first_href
                )
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a.MuiPaper-root')))
            except TimeoutException:
                logging.error("Timeout waiting for the next page to load. Stopping.")
                break
                
            current_page += 1
            
    except KeyboardInterrupt:
        logging.info("Scraping manually interrupted by user.")
    except Exception as e:
        logging.error(f"Unexpected error: {e}")
    finally:
        driver.quit()

    # บันทึกเฉพาะ URL
    with OUTPUT_CSV_FILE.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['ListingURL'])
        for u in sorted(all_urls):
            writer.writerow([u])
            
    logging.info(f"Successfully saved {len(all_urls)} URLs to {OUTPUT_CSV_FILE}")

if __name__ == "__main__":
    main()

2026-03-29 15:23:19,249 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 15:23:21,602 - INFO - Starting scrape at: https://www.9asset.com/%E0%B8%82%E0%B8%B2%E0%B8%A2/%E0%B8%AD%E0%B8%AA%E0%B8%B1%E0%B8%87%E0%B8%AB%E0%B8%B2/%E0%B8%81%E0%B8%A3%E0%B8%B8%E0%B8%87%E0%B9%80%E0%B8%97%E0%B8%9E%E0%B8%AF
2026-03-29 15:23:29,969 - INFO - Waiting 40 seconds for manual login...
2026-03-29 15:24:09,970 - INFO - Processing Page 1/20...
2026-03-29 15:24:11,785 - INFO - Found 24 listings (Total unique: 24)
2026-03-29 15:24:14,565 - INFO - Processing Page 2/20...
2026-03-29 15:24:15,602 - INFO - Found 24 listings (Total unique: 48)
2026-03-29 15:24:18,445 - INFO - Processing Page 3/20...
2026-03-29 15:24:19,839 - INFO - Found 24 listings (Total unique: 72)
2026-03-29 15:24:22,141 - INFO - Processing Page 4/20...
2026-03-29 15:24:23,603 - INFO - Found 24 listings (Total unique: 96)
2026-03-29 15:24:26,403 - INFO - Processing Page 5/20.

In [8]:
import csv
import time
import logging
from pathlib import Path
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/9asset')
INPUT_CSV = BASE_DIR / '9asset_listing_urls.csv'
OUTPUT_CSV = BASE_DIR / '9asset_full_details.csv'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def setup_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
    }
    options.add_experimental_option("prefs", prefs)
    
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    
    return uc.Chrome(options=options, version_main=145)

def extract_content(driver: uc.Chrome) -> str:
    data = []
    try:
        h1 = driver.find_element(By.TAG_NAME, 'h1')
        data.extend(["--- Listing Title ---", h1.text.strip()])
        
        try:
            loc = driver.find_element(By.XPATH, "//h1/following-sibling::p")
            data.append(f"Location: {loc.text.strip()}")
        except NoSuchElementException:
            pass
        
        prices = driver.find_elements(By.XPATH, "//p[text()='ขาย' or text()='ให้เช่า' or text()='จำนองขายฝาก' or text()='เซ้ง']/parent::div")
        if prices:
            data.append("\n--- Price Information ---")
            data.extend([' '.join(p.text.split()) for p in prices if p.text.strip()])

        data.append("\n--- Property Info ---")
        
        top_details = driver.find_elements(By.XPATH, "//div[p[contains(@class, 'MuiTypography-alignCenter')]]")
        for d in top_details:
            parts = [p.text.strip() for p in d.find_elements(By.TAG_NAME, 'p') if p.text.strip()]
            if len(parts) >= 2:
                data.append(f"{parts[-1]}: {' '.join(parts[:-1])}")
                
        list_details = driver.find_elements(By.XPATH, "//div[p[contains(@class, 'css-1c90359')]]")
        for d in list_details:
            parts = [p.text.strip() for p in d.find_elements(By.TAG_NAME, 'p') if p.text.strip()]
            if len(parts) >= 2:
                data.append(f"{parts[0]}: {' '.join(parts[1:])}")

        try:
            btn = driver.find_element(By.XPATH, "//button[contains(., 'อ่านทั้งหมด')]")
            driver.execute_script("arguments[0].click();", btn)
        except NoSuchElementException:
            pass

        try:
            desc = driver.find_element(By.XPATH, "//h2[contains(text(), 'คำอธิบาย')]/following-sibling::div").text
            clean_desc = desc.replace("อ่านน้อยลง", "").strip()
            if clean_desc:
                data.extend(["\n--- Description ---", clean_desc])
        except NoSuchElementException:
            pass

    except Exception:
        pass

    return "\n".join(data)

def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if not INPUT_CSV.exists():
        logging.warning(f"File not found: {INPUT_CSV}")
        return

    with INPUT_CSV.open('r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None)
        urls = [row[0] for row in reader if row]

    if not urls:
        logging.info("No URLs found to scrape.")
        return

    driver = setup_driver()
    wait = WebDriverWait(driver, 10)

    try:
        driver.get(urls[0])
        logging.info("Waiting 40 seconds for manual login...")
        time.sleep(40)

        with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Post_URL', 'Full_Content'])

            for i, url in enumerate(urls, 1):
                try:
                    if driver.current_url != url:
                        driver.get(url)
                    
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, 'h1')))
                    
                    content = extract_content(driver)
                    writer.writerow([url, content])
                    
                    if i % 10 == 0:
                        logging.info(f"Progress: [{i}/{len(urls)}] scraped.")

                except TimeoutException:
                    logging.error(f"[{i}/{len(urls)}] Timeout loading: {url}")
                except Exception as e:
                    logging.error(f"[{i}/{len(urls)}] Error scraping {url}: {e}")

    finally:
        driver.quit()
        logging.info(f"Scraping completed. Saved to: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

2026-03-29 15:26:56,459 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 15:27:02,603 - INFO - Waiting 40 seconds for manual login...
2026-03-29 15:28:23,185 - INFO - Progress: [10/480] scraped.
2026-03-29 15:29:17,938 - INFO - Progress: [20/480] scraped.
2026-03-29 15:30:05,679 - INFO - Progress: [30/480] scraped.
2026-03-29 15:31:00,044 - INFO - Progress: [40/480] scraped.
2026-03-29 15:31:53,542 - INFO - Progress: [50/480] scraped.
2026-03-29 15:32:45,285 - INFO - Progress: [60/480] scraped.
2026-03-29 15:33:34,879 - INFO - Progress: [70/480] scraped.
2026-03-29 15:34:19,786 - INFO - Progress: [80/480] scraped.
2026-03-29 15:34:57,509 - INFO - Progress: [90/480] scraped.
2026-03-29 15:35:41,974 - INFO - Progress: [100/480] scraped.
2026-03-29 15:37:05,988 - INFO - Progress: [110/480] scraped.
2026-03-29 15:38:03,073 - INFO - Progress: [120/480] scraped.
2026-03-29 15:38:46,181 - INFO - Progress: [130/480] scrape